# Verify headline embeddings

Re-embed a few months of RavenPack headlines with RavenBERT and compare against the
migrated `headline_embeddings` artifact (originally produced by the legacy
`embed_headlines.py` in the Laboratory repo). Same model ⇒ per-story cosine similarity
should be ≈ 1.0 everywhere.

Embeddings are L2-normalized, so cosine similarity is a plain dot product.

In [ ]:
# Cell 1 -- setup, env, resolve artifact paths
import os
from pathlib import Path

import numpy as np
import polars as pl
from dotenv import find_dotenv, load_dotenv

from datalake import DatalakeIndex

load_dotenv(find_dotenv(usecwd=True))
DATALAKE_ROOT = os.environ["DATALAKE_ROOT"]
MODEL_PATH = os.environ["RAVENBERT_EMBEDDING_MODEL_PATH"]
MONTHS = ["2000-01", "2000-02", "2000-03"]

with DatalakeIndex(DATALAKE_ROOT) as dl:
    ingest = dl.latest("ravenpack_headlines")
    embeds = dl.latest("headline_embeddings", model="ravenbert", version="1.0")

ingest_dir = ingest.path
embeds_dir = embeds.path
print("ingest artifact :", ingest.artifact_id)
print("               ->", ingest_dir)
print("embeds artifact :", embeds.artifact_id)
print("               ->", embeds_dir)
print("model           :", MODEL_PATH)

In [ ]:
# Cell 2 -- embed MONTHS fresh, join with stored embeddings on RP_STORY_ID
from ravenbert.embedding.model import EmbeddingModel

model = EmbeddingModel.from_path(MODEL_PATH)

fresh_frames, stored_frames = [], []
for month in MONTHS:
    src = pl.read_parquet(ingest_dir / f"{month}.parquet", columns=["RP_STORY_ID", "HEADLINE"])
    headlines = [h if h is not None else "" for h in src["HEADLINE"].to_list()]
    vecs = np.asarray(model.encode(headlines), dtype=np.float32)
    fresh_frames.append(pl.DataFrame({"RP_STORY_ID": src["RP_STORY_ID"], "FRESH": list(vecs)}))
    stored_frames.append(
        pl.read_parquet(embeds_dir / f"{month}.parquet", columns=["RP_STORY_ID", "EMBEDDING"])
    )

fresh = pl.concat(fresh_frames)
stored = pl.concat(stored_frames)
joined = fresh.join(stored, on="RP_STORY_ID", how="inner")
print(f"fresh={fresh.height}  stored={stored.height}  joined={joined.height}")
assert joined.height > 0, "no overlap on RP_STORY_ID"

In [ ]:
# Cell 3 -- per-story cosine similarity (normalized vectors => dot product)
fresh_mat = np.asarray(joined["FRESH"].to_list(), dtype=np.float32)
stored_mat = np.asarray(joined["EMBEDDING"].to_list(), dtype=np.float32)

# Renormalize defensively; float16 storage introduces a tiny norm drift.
fresh_mat /= np.linalg.norm(fresh_mat, axis=1, keepdims=True)
stored_mat /= np.linalg.norm(stored_mat, axis=1, keepdims=True)

similarity = np.einsum("ij,ij->i", fresh_mat, stored_mat)
sim = joined.select("RP_STORY_ID").with_columns(pl.Series("similarity", similarity))
sim.head()

In [ ]:
# Cell 4 -- summary stats + histogram
import matplotlib.pyplot as plt

print(f"n        = {similarity.size}")
print(f"mean     = {similarity.mean():.6f}")
print(f"std      = {similarity.std():.6f}")
print(f"min      = {similarity.min():.6f}")
print(f"max      = {similarity.max():.6f}")
print(f"p01      = {np.percentile(similarity, 1):.6f}")

plt.figure(figsize=(7, 4))
plt.hist(np.clip(similarity, 0.9, 1.0), bins=100)
plt.xlabel("cosine similarity (fresh vs stored)")
plt.ylabel("stories")
plt.title(f"RavenBERT re-embed vs migrated archive ({', '.join(MONTHS)})")
plt.yscale("log")
plt.show()

In [ ]:
# Cell 5 -- flag suspicious stories (similarity < 0.99)
THRESHOLD = 0.99
suspicious = sim.filter(pl.col("similarity") < THRESHOLD).sort("similarity")
print(f"{suspicious.height} / {sim.height} stories below {THRESHOLD} "
      f"({100 * suspicious.height / sim.height:.4f}%)")

if suspicious.height:
    ctx = pl.concat(
        [pl.read_parquet(ingest_dir / f"{m}.parquet", columns=["RP_STORY_ID", "HEADLINE"])
         for m in MONTHS]
    )
    display(suspicious.join(ctx, on="RP_STORY_ID", how="left").head(25))
else:
    print("OK -- all similarities >= 0.99, consistent with the same model.")